# V6.2E Localization Decomposition + Canonical ROI Hypothesis Audit

**No training. VAL only. TEST is not predicted/evaluated.**

This notebook tests the exact hypothesis:

> The detector may already find the correct defect location, but strict / tiny / inconsistent
> bbox extent makes IoU collapse at higher thresholds. A canonical defect ROI with a moderate
> minimum context window may therefore be a better annotation target than an ultra-tight box.

It does four things:

1. restore the accepted V6.2E lineage cleanly;
2. restore the completed **YOLO11m @1024** checkpoint;
3. decompose localization error into **center error vs extent/size error**;
4. sweep hypothetical canonical ROI rules such as minimum 16/20/24/32/48/64 px at the
   1024 inference scale, with modest bbox expansion.

Important: the canonical ROI sweep is a **diagnostic upper-bound / what-if experiment**.
It is not reported as mAP and it does not mutate V6.2E.


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, hashlib, zipfile, math, re, time
from collections import Counter, defaultdict

W = Path("/kaggle/working")
TMP = Path("/tmp/v6p2e_localization_roi_audit_clean")
BASE_EXTRACT = TMP/"base"
OVERLAY_EXTRACT = TMP/"overlay"
RECON_LABELS = TMP/"reconciled_labels"
OUT = W/"v6p2e_localization_roi_audit"
REPORT = OUT/"report"
PREVIEW = OUT/"previews"

for p in [TMP, BASE_EXTRACT, OVERLAY_EXTRACT, RECON_LABELS, OUT, REPORT, PREVIEW]:
    p.mkdir(parents=True, exist_ok=True)

FINAL_TAG = "v6p2e_23562a45b343"
PARENT_TAG = "v6p1_5d0b1149891c"
BASE_TAG = "56b7ea194ee5"
OVERLAY_TAG = "ee2d3c4497fd"

DATA_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
FINAL_ROOT = f"{DATA_ROOT}/v6p2e/{FINAL_TAG}"
PARENT_ROOT = f"{DATA_ROOT}/v6p1/{PARENT_TAG}"

BASE_STASH = f"{DATA_ROOT}/curated_v6_rs1280_{BASE_TAG}.tar"
OVERLAY_STASH = f"{PARENT_ROOT}/v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"
FREEZE_STASH = f"{FINAL_ROOT}/v6p2e.freeze.json"
MANIFEST_STASH = f"{FINAL_ROOT}/v6p2e_split_manifest.csv"
RECON_STASH = f"{FINAL_ROOT}/v6p2e_reconciled_labels.zip"

NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

EXPECTED_SPLITS = {"train":17483, "val":2272, "test":2012}
EXPECTED_VAL_BOXES = [778,791,163,821,98,252]

CKPT_ROOT = (
    "stash://bunpmc/projects/yolov11sdi/checkpoints/"
    f"v6p2e/{FINAL_TAG}/yolo_baselines"
)
MODEL_REMOTE = f"{CKPT_ROOT}/yolo11m_v6p2e_1024/best.pt"

REMOTE_REPORT = (
    "stash://bunpmc/projects/yolov11sdi/reports/"
    f"v6p2e/{FINAL_TAG}/localization_canonical_roi_audit_v1"
)

IMG_EXTS={".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

# Inference is intentionally permissive so that "correct-center but bad-extent"
# candidates are not discarded by confidence thresholding.
IMGSZ = 1024
PRED_CONF = 0.01
PRED_NMS_IOU = 0.70
MAX_DET = 300
DEVICE = 0

# Matching gate for diagnostic pairing.
# A low-IoU prediction can still be paired if the center is reasonably close.
MAX_CENTER_DIAG_FOR_MATCH = 1.50

# User's "around 20 px" idea is explicitly included in this sweep.
MIN_SIDE_INPUT_PX = [16, 20, 24, 32, 48, 64]
EXPAND_FACTORS = [1.00, 1.20, 1.40, 1.60]

ROI_CLASSES = [
    "insulator_broken",
    "insulator_flashover",
    "broken_strand",
]

SEED = 42


## 1. Runtime + Camber

In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics", "pandas", "pyyaml", "pillow", "scipy"
])

import numpy as np
import pandas as pd
import yaml
import torch
from PIL import Image, ImageDraw, ImageOps
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("CAMBER_API_KEY")
if not key:
    raise RuntimeError("Missing CAMBER_API_KEY")
os.environ["CAMBER_API_KEY"] = key

if shutil.which("camber") is None and not (Path.home()/".camber/bin/camber").exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:"
    f"{Path.home()}/.local/bin:"
    + os.environ.get("PATH","")
)
CAMBER = shutil.which("camber") or str(Path.home()/".camber/bin/camber")

def stash_cp(src, dst, required=True):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(src), str(dst)],
        text=True, capture_output=True, env=os.environ.copy()
    )
    ok = r.returncode == 0 and dst.exists()
    if required and not ok:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    return ok

def stash_put(src, dst):
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(src), str(dst)],
        text=True, capture_output=True, env=os.environ.copy()
    )
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    print("Uploaded:", dst)

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.device_count():
    print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "Enable a Kaggle GPU."


## 2. Clean restore accepted V6.2E

This deliberately avoids stale `/tmp` extraction. Only **TRAIN + VAL** records are resolved
for this notebook. TEST metadata remains in the freeze, but TEST images are not predicted.


In [ ]:
freeze_local = TMP/"v6p2e.freeze.json"
manifest_local = TMP/"v6p2e_split_manifest.csv"
recon_zip = TMP/"v6p2e_reconciled_labels.zip"
base_tar = TMP/f"base_{BASE_TAG}.tar"
overlay_tar = TMP/f"overlay_{OVERLAY_TAG}.tar.gz"

# IMPORTANT:
# Never reuse previously extracted /tmp source trees. Earlier V6.3 notebook
# versions used "if data.yaml exists -> reuse", which can preserve a stale or
# partial extraction across reruns in the same Kaggle session.
for p in [BASE_EXTRACT, OVERLAY_EXTRACT, RECON_LABELS]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

for p in [freeze_local, manifest_local, recon_zip, base_tar, overlay_tar]:
    p.unlink(missing_ok=True)

print("Downloading accepted V6.2E metadata...")
stash_cp(FREEZE_STASH, freeze_local)
stash_cp(MANIFEST_STASH, manifest_local)
stash_cp(RECON_STASH, recon_zip)

freeze = json.loads(freeze_local.read_text())
manifest = pd.read_csv(manifest_local)

assert freeze["final_tag"] == FINAL_TAG
assert freeze["freeze_ready"] is True
assert len(manifest) == sum(freeze["split_counts"].values()), (
    len(manifest), freeze["split_counts"]
)

print("Downloading immutable source archives from Camber...")
stash_cp(BASE_STASH, base_tar)
stash_cp(OVERLAY_STASH, overlay_tar)

# Archive sanity before extraction.
def tar_names(path, gz=False):
    flag = "-tzf" if gz else "-tf"
    r = subprocess.run(
        ["tar", flag, str(path)],
        text=True, capture_output=True
    )
    if r.returncode != 0:
        raise RuntimeError(
            f"Cannot list archive {path}\n{r.stderr[-2000:]}"
        )
    return r.stdout.splitlines()

base_names = tar_names(base_tar, gz=False)
overlay_names = tar_names(overlay_tar, gz=True)

print("BASE archive entries:", len(base_names))
print("OVERLAY archive entries:", len(overlay_names))

# This exact file is the one that exposed stale/mismatched extraction earlier.
# Do not require the basename to exist because a previous curation stage may
# legitimately rename files; this is only a diagnostic.
_known_problem_basename = "fotl_image_val159_6530eaae.jpg"
print(
    "Known-problem basename present in raw OVERLAY archive:",
    any(Path(x).name == _known_problem_basename for x in overlay_names)
)

print("Extracting clean source trees...")
subprocess.check_call(["tar","-xf",str(base_tar),"-C",str(BASE_EXTRACT)])
subprocess.check_call(["tar","-xzf",str(overlay_tar),"-C",str(OVERLAY_EXTRACT)])

# Reconciled labels are also rebuilt cleanly.
with zipfile.ZipFile(recon_zip) as z:
    z.extractall(RECON_LABELS)

def norm_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x:int(x))]
    return list(names)

def find_root(root):
    matches=[]
    for yp in Path(root).rglob("data.yaml"):
        try:
            cfg = yaml.safe_load(yp.read_text())
            if norm_names(cfg.get("names",[])) == NAMES:
                matches.append(yp.parent)
        except Exception:
            pass
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one six-class dataset root under {root}; "
            f"found {len(matches)}: {matches}"
        )
    return matches[0]

BASE = find_root(BASE_EXTRACT)
OVERLAY = find_root(OVERLAY_EXTRACT)

print("Freeze:", FINAL_TAG)
print("Manifest rows:", len(manifest))
print("BASE:", BASE)
print("OVERLAY:", OVERLAY)
print("Manifest columns:", list(manifest.columns))
manifest_dev = manifest[manifest["split"].isin(["train","val"])].copy()
print("Development-only manifest rows:", len(manifest_dev))
assert len(manifest_dev) == EXPECTED_SPLITS["train"] + EXPECTED_SPLITS["val"]


# Do not retain archive files after a successful extraction.
base_tar.unlink(missing_ok=True)
overlay_tar.unlink(missing_ok=True)
recon_zip.unlink(missing_ok=True)


## 3. Resolve exact physical source bytes by SHA256

In [ ]:
PROVENANCE_COL_CANDIDATES = [
    "source_dataset", "original_source", "dataset", "dataset_source",
    "raw_source", "provenance", "source_name",
]

IMG_EXTS={".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

def sha256_file(path, chunk=4*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def explicit_provenance(row):
    for c in PROVENANCE_COL_CANDIDATES:
        if c in manifest.columns:
            v = str(row.get(c,"")).strip()
            if v and v.lower() not in {"nan","none",""}:
                return v, f"manifest:{c}"
    return None, None

def parse_old_path(row):
    old_path = str(row["image_path_source"]).replace("\\","/")
    old_split = str(row["previous_v6p1_split"])

    # Prefer the declared split marker, but tolerate rows whose logical split lineage
    # no longer matches the physical archive split.
    marker = f"/images/{old_split}/"
    if marker in old_path:
        suffix = old_path.split(marker,1)[1]
    else:
        m = re.search(r"/images/(train|val|test)/(.*)$", old_path)
        if not m:
            raise RuntimeError(f"Cannot parse source path: {old_path}")
        suffix = m.group(2)

    if suffix.startswith("base/"):
        family = "base"
        rel = suffix[len("base/"):]
        preferred_root = BASE
    elif suffix.startswith("fotl/"):
        family = "fotl"
        rel = suffix[len("fotl/"):]
        preferred_root = OVERLAY
    else:
        raise RuntimeError(f"Unknown source suffix: {suffix}")

    parts = Path(rel).parts
    path_group = parts[0] if len(parts) > 1 else family
    return old_path, old_split, family, rel, preferred_root, path_group

# Build cheap basename indexes once for BOTH immutable source artifacts.
ROOT_NAME = {
    str(BASE): "base",
    str(OVERLAY): "fotl",
}
ALL_ROOTS=[BASE,OVERLAY]

BASENAME_INDEX=defaultdict(list)
for rr in ALL_ROOTS:
    img_root=Path(rr)/"images"
    if not img_root.exists():
        continue
    for p in img_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            BASENAME_INDEX[p.name].append((Path(rr),p))

print("Indexed physical source images by basename:",
      sum(len(v) for v in BASENAME_INDEX.values()))

_SHA_INDEX=None

def build_sha_index():
    """
    Expensive fallback, built at most once.
    It scans both BASE and OVERLAY and maps SHA256 -> actual image path/root.
    This handles artifacts where a manifest's original filename/path is no longer
    present but the exact image bytes survived under a renamed path.
    """
    global _SHA_INDEX
    if _SHA_INDEX is not None:
        return _SHA_INDEX

    idx=defaultdict(list)
    n=0
    for rr in ALL_ROOTS:
        img_root=Path(rr)/"images"
        if not img_root.exists():
            continue
        for p in img_root.rglob("*"):
            if not (p.is_file() and p.suffix.lower() in IMG_EXTS):
                continue
            idx[sha256_file(p)].append((Path(rr),p))
            n+=1
            if n%5000==0:
                print("SHA-indexed",n,"source images...")

    _SHA_INDEX=idx
    print("✅ SHA index built:",n,"images,",len(idx),"unique hashes")
    return _SHA_INDEX

def label_for_actual_image(actual_root, ip, rel_hint=None):
    actual_root=Path(actual_root)
    ip=Path(ip)

    try:
        rel_from_images=ip.relative_to(actual_root/"images")
        lp=(actual_root/"labels"/rel_from_images).with_suffix(".txt")
        if lp.exists():
            return lp, str(rel_from_images.parts[0])
    except Exception:
        rel_from_images=None

    # Fallback by basename.
    candidates=list((actual_root/"labels").rglob(ip.stem+".txt"))
    if rel_hint is not None:
        hint_name=Path(rel_hint).with_suffix(".txt").name
        hinted=[p for p in candidates if p.name==hint_name]
        if len(hinted)==1:
            candidates=hinted

    if len(candidates)==1:
        lp=candidates[0]
        try:
            physical_split=str(lp.relative_to(actual_root/"labels").parts[0])
        except Exception:
            physical_split="unknown"
        return lp, physical_split

    # We still return a derived path if possible so caller can emit a precise error.
    if rel_from_images is not None:
        return (actual_root/"labels"/rel_from_images).with_suffix(".txt"), str(rel_from_images.parts[0])

    return actual_root/"labels"/(ip.stem+".txt"), "unknown"

def resolve_physical_pair(preferred_root, old_split, rel, expected_sha):
    """
    Resolve exact source bytes with progressively broader search:

      A. preferred root + declared split
      B. preferred root + any split
      C. same basename across BOTH BASE and OVERLAY
      D. exact SHA256 across BOTH BASE and OVERLAY, even if renamed

    A file is accepted only when SHA256 == manifest SHA256.
    Returns:
      actual_root, image_path, label_path, physical_source_split, resolution_mode
    """
    preferred_root=Path(preferred_root)
    rel=Path(rel)

    candidates=[]
    seen=set()

    def add(actual_root,p,mode):
        actual_root=Path(actual_root)
        p=Path(p)
        key=(str(actual_root),str(p))
        if key not in seen:
            seen.add(key)
            candidates.append((actual_root,p,mode))

    # A: exact manifest lineage path.
    add(preferred_root, preferred_root/"images"/old_split/rel, "preferred_declared_split")

    # B: same relative path in any split of preferred root.
    for sp in ["train","val","test"]:
        add(preferred_root, preferred_root/"images"/sp/rel, "preferred_other_split")

    # C: basename across both artifacts.
    for actual_root,p in BASENAME_INDEX.get(rel.name,[]):
        add(actual_root,p,"cross_root_same_basename")

    existing=[(rr,p,mode) for rr,p,mode in candidates if p.exists() and p.is_file()]

    for actual_root,ip,mode in existing:
        if sha256_file(ip)==expected_sha:
            lp,physical_split=label_for_actual_image(actual_root,ip,rel)
            return actual_root,ip,lp,physical_split,mode

    # D: Exact-byte fallback across both artifacts.
    sha_index=build_sha_index()
    matches=sha_index.get(expected_sha,[])
    if matches:
        # Prefer the manifest-indicated artifact if the hash exists in both.
        matches=sorted(matches,key=lambda x: 0 if Path(x[0])==preferred_root else 1)
        actual_root,ip=matches[0]
        lp,physical_split=label_for_actual_image(actual_root,ip,rel)
        return actual_root,ip,lp,physical_split,"cross_root_sha256"

    # At this point we have freshly downloaded + freshly extracted both immutable
    # source archives in this same run, and searched them by exact SHA256.
    # Therefore this is a genuine artifact/manifest mismatch, not a path-cache issue.
    missing_row = {
        "expected_sha": expected_sha,
        "preferred_root": str(preferred_root),
        "old_split": old_split,
        "relative_path": str(rel),
        "same_basename_candidates": [
            str(p) for _,p in BASENAME_INDEX.get(rel.name,[])[:20]
        ],
    }
    diag = OUT/"MISSING_SOURCE_BYTES.json"
    diag.write_text(json.dumps(missing_row, indent=2))
    raise FileNotFoundError(
        "CLEAN-RESTORE INTEGRITY FAILURE.\n"
        "The exact manifest SHA256 is absent from BOTH freshly restored source artifacts.\n"
        f"Diagnostic written to: {diag}\n"
        + json.dumps(missing_row, indent=2)
    )

rows=[]

for idx,row in manifest_dev.iterrows():
    old_path,old_split,family,rel,preferred_root,path_group=parse_old_path(row)
    expected_sha=str(row["sha256"])

    prov,prov_basis=explicit_provenance(row)
    if prov is None:
        if path_group not in {"base","fotl"} and len(Path(rel).parts)>1:
            prov=path_group
            prov_basis="path_subdir"
        else:
            prov=family
            prov_basis="coarse_family_only"

    actual_root,ip,original_lp,physical_source_split,resolution_mode=resolve_physical_pair(
        preferred_root=preferred_root,
        old_split=old_split,
        rel=rel,
        expected_sha=expected_sha,
    )

    # If the manifest says this sample used a reconciled label, that label wins.
    rp=RECON_LABELS/f"{expected_sha}.txt"
    if str(row.get("resolution","")).startswith("merged"):
        if not rp.exists():
            raise FileNotFoundError(f"Missing reconciled label: {rp}")
        lp=rp
    else:
        lp=original_lp
        if not lp.exists():
            raise FileNotFoundError(
                "Resolved exact image bytes but corresponding label is missing.\n"
                f"actual_root={actual_root}\n"
                f"image={ip}\n"
                f"derived_label={lp}\n"
                f"sha={expected_sha}\n"
                f"resolution_mode={resolution_mode}"
            )

    rows.append({
        "row_id":idx,
        "sha256":expected_sha,
        "split":str(row["split"]),
        "previous_split":old_split,
        "physical_source_split":physical_source_split,
        "coarse_family":family,
        "actual_artifact":ROOT_NAME.get(str(actual_root),str(actual_root)),
        "provenance":str(prov),
        "provenance_basis":prov_basis,
        "relative_path":rel,
        "image_path":str(ip),
        "label_path":str(lp),
        "path_resolution_mode":resolution_mode,
        "resolution":str(row.get("resolution","")),
    })

rec=pd.DataFrame(rows)

if (rec.provenance_basis=="coarse_family_only").mean()>0.80:
    print("⚠️ Most original dataset provenance is no longer recoverable from this artifact.")
    print("   This audit can still run, but source-by-source conclusions will be limited.")

print("\nResolution modes:")
display(rec["path_resolution_mode"].value_counts().rename_axis("mode").reset_index(name="images"))

cross_artifact=rec[
    ((rec.coarse_family=="fotl") & (rec.actual_artifact!="fotl")) |
    ((rec.coarse_family=="base") & (rec.actual_artifact!="base"))
]
print("Cross-artifact recoveries:",len(cross_artifact))
if len(cross_artifact):
    display(cross_artifact[
        ["coarse_family","actual_artifact","previous_split",
         "physical_source_split","relative_path","path_resolution_mode"]
    ].head(50))

split_mismatch=rec[rec["previous_split"]!=rec["physical_source_split"]]
print("Physical split relocations resolved:",len(split_mismatch))
if len(split_mismatch):
    display(split_mismatch[
        ["coarse_family","actual_artifact","previous_split",
         "physical_source_split","relative_path","path_resolution_mode"]
    ].head(50))

display(
    rec.groupby(["provenance","provenance_basis","split"])
       .size().reset_index(name="images")
       .sort_values("images",ascending=False)
       .head(100)
)

rec.to_csv(OUT/"recovered_provenance.csv",index=False)


## 4. Parse TRAIN/VAL labels and verify VAL distribution

In [ ]:
def parse_label(path):
    boxes = []
    for line_no,line in enumerate(Path(path).read_text(errors="ignore").splitlines(),1):
        if not line.strip():
            continue
        p = line.split()
        if len(p) < 5:
            raise RuntimeError(f"Malformed label: {path}:{line_no}")
        cid = int(float(p[0]))
        x,y,w,h = map(float,p[1:5])
        boxes.append({
            "line_no":line_no,
            "class_id":cid,
            "class":NAMES[cid],
            "cx":x,"cy":y,"w":w,"h":h,
        })
    return boxes

box_rows = []
image_rows = []

for _,r in rec.iterrows():
    boxes = parse_label(r.label_path)
    image_rows.append({**r.to_dict(), "n_boxes":len(boxes)})

    with Image.open(r.image_path) as im:
        Wd,Hd = im.size

    for b in boxes:
        area = b["w"]*b["h"]
        ar = max(b["w"]/b["h"], b["h"]/b["w"]) if b["w"]>0 and b["h"]>0 else np.inf
        box_rows.append({
            **r.to_dict(),
            **b,
            "area":area,
            "aspect_ratio":ar,
            "pixel_w":b["w"]*Wd,
            "pixel_h":b["h"]*Hd,
        })

images_df = pd.DataFrame(image_rows)
boxes_df = pd.DataFrame(box_rows)

print("Images:",len(images_df))
print("Boxes:",len(boxes_df))
display(boxes_df.groupby(["provenance","class"]).size().reset_index(name="boxes"))


In [ ]:
val_boxes = boxes_df[boxes_df["split"]=="val"].copy()
val_images = images_df[images_df["split"]=="val"].copy()

got_val = (
    val_boxes.groupby("class_id").size()
    .reindex(range(len(NAMES)), fill_value=0)
    .astype(int).tolist()
)

print("VAL images:", len(val_images))
print("VAL boxes :", len(val_boxes))
print("VAL boxes/class:", got_val)

assert len(val_images) == EXPECTED_SPLITS["val"]
assert got_val == EXPECTED_VAL_BOXES, (got_val, EXPECTED_VAL_BOXES)
print("✅ Accepted VAL distribution reproduced. TEST not evaluated.")


## 5. Restore completed YOLO11m @1024 checkpoint

In [ ]:
MODEL_LOCAL = TMP/"yolo11m_v6p2e_1024_best.pt"
stash_cp(MODEL_REMOTE, MODEL_LOCAL, required=True)

def sha256_file(path, chunk=4*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

MODEL_SHA = sha256_file(MODEL_LOCAL)
print("Model:", MODEL_LOCAL)
print("SHA256:", MODEL_SHA)

model = YOLO(str(MODEL_LOCAL))


## 6. Build pixel-space VAL ground truth

Ultralytics predictions are returned in original-image coordinates, so GT is converted to the
same coordinate system. We also record the 1024-letterbox scale so a "20 px minimum ROI"
means approximately 20 px in model-input space rather than 20 arbitrary source-image pixels.


In [ ]:
def xywhn_to_xyxy_px(cx,cy,w,h,Wd,Hd):
    x1=(cx-w/2)*Wd
    y1=(cy-h/2)*Hd
    x2=(cx+w/2)*Wd
    y2=(cy+h/2)*Hd
    return x1,y1,x2,y2

gt_rows=[]

for _,imgrow in val_images.iterrows():
    with Image.open(imgrow.image_path) as im:
        Wd,Hd=im.size

    scale_1024=min(IMGSZ/Wd, IMGSZ/Hd)

    b=val_boxes[val_boxes.sha256==imgrow.sha256]
    for _,r in b.iterrows():
        x1,y1,x2,y2=xywhn_to_xyxy_px(
            r.cx,r.cy,r.w,r.h,Wd,Hd
        )
        gt_rows.append({
            "sha256":imgrow.sha256,
            "image_path":imgrow.image_path,
            "class_id":int(r.class_id),
            "class":r["class"],
            "gt_x1":x1,"gt_y1":y1,"gt_x2":x2,"gt_y2":y2,
            "image_w":Wd,"image_h":Hd,
            "scale_1024":scale_1024,
            "gt_w":x2-x1,
            "gt_h":y2-y1,
            "gt_area":max(0,x2-x1)*max(0,y2-y1),
        })

gt=pd.DataFrame(gt_rows)
print("GT rows:",len(gt))
assert len(gt)==sum(EXPECTED_VAL_BOXES)
gt.to_csv(REPORT/"val_gt.csv.gz",index=False,compression="gzip")


## 7. Run / restore permissive VAL predictions

Predictions are cached on Camber. A rerun with the same dataset tag, model SHA, inference size,
and confidence threshold can skip the expensive inference pass.


In [ ]:
PRED_CACHE = REPORT/"val_predictions_yolo11m1024_conf001.csv.gz"
PRED_META = REPORT/"val_predictions_meta.json"

cache_sig = {
    "dataset_tag":FINAL_TAG,
    "model_sha256":MODEL_SHA,
    "imgsz":IMGSZ,
    "conf":PRED_CONF,
    "nms_iou":PRED_NMS_IOU,
    "max_det":MAX_DET,
    "split":"val",
}

REMOTE_PRED = f"{REMOTE_REPORT}/{PRED_CACHE.name}"
REMOTE_META = f"{REMOTE_REPORT}/{PRED_META.name}"

cache_ok=False
remote_meta_tmp=TMP/"remote_pred_meta.json"

if stash_cp(REMOTE_META, remote_meta_tmp, required=False):
    try:
        old_meta=json.loads(remote_meta_tmp.read_text())
        if old_meta==cache_sig and stash_cp(REMOTE_PRED,PRED_CACHE,required=False):
            cache_ok=True
            print("✅ Reusing cached VAL predictions from Camber.")
    except Exception:
        pass

if cache_ok:
    pred=pd.read_csv(PRED_CACHE)
else:
    paths=val_images["image_path"].tolist()
    sha_order=val_images["sha256"].tolist()

    pred_rows=[]
    t0=time.time()

    results=model.predict(
        source=paths,
        imgsz=IMGSZ,
        conf=PRED_CONF,
        iou=PRED_NMS_IOU,
        max_det=MAX_DET,
        device=DEVICE,
        stream=True,
        verbose=False,
    )

    for idx,(sha,res) in enumerate(zip(sha_order,results),1):
        if res.boxes is not None and len(res.boxes):
            xyxy=res.boxes.xyxy.detach().cpu().numpy()
            cls=res.boxes.cls.detach().cpu().numpy().astype(int)
            conf=res.boxes.conf.detach().cpu().numpy()

            for j in range(len(cls)):
                pred_rows.append({
                    "sha256":sha,
                    "class_id":int(cls[j]),
                    "class":NAMES[int(cls[j])],
                    "conf":float(conf[j]),
                    "pred_x1":float(xyxy[j,0]),
                    "pred_y1":float(xyxy[j,1]),
                    "pred_x2":float(xyxy[j,2]),
                    "pred_y2":float(xyxy[j,3]),
                })

        if idx%250==0:
            print(f"Predicted {idx}/{len(paths)} images")

    pred=pd.DataFrame(pred_rows)
    pred.to_csv(PRED_CACHE,index=False,compression="gzip")
    PRED_META.write_text(json.dumps(cache_sig,indent=2))

    print("Inference seconds:",round(time.time()-t0,1))
    print("Predictions:",len(pred))

    stash_put(PRED_CACHE,REMOTE_PRED)
    stash_put(PRED_META,REMOTE_META)

display(pred.groupby("class").size().reset_index(name="predictions"))


## 8. Localization decomposition + center-aware one-to-one matching

For each image/class we use a Hungarian assignment whose cost is dominated by normalized
center distance, with IoU as a secondary term. This is intentional: an object can still be
paired for diagnosis even when bbox extent is poor.

For every matched GT we compute:

- `actual_iou`: real predicted box vs GT;
- `center_only_iou`: predicted center but **GT width/height**. This isolates center accuracy;
- `extent_only_iou`: GT center but **predicted width/height**. This isolates extent/size/shape;
- width / height / area ratios;
- normalized center error relative to GT diagonal.


In [ ]:
def box_iou_xyxy(a,b):
    ax1,ay1,ax2,ay2=map(float,a)
    bx1,by1,bx2,by2=map(float,b)
    ix1=max(ax1,bx1); iy1=max(ay1,by1)
    ix2=min(ax2,bx2); iy2=min(ay2,by2)
    iw=max(0.0,ix2-ix1); ih=max(0.0,iy2-iy1)
    inter=iw*ih
    aa=max(0.0,ax2-ax1)*max(0.0,ay2-ay1)
    bb=max(0.0,bx2-bx1)*max(0.0,by2-by1)
    den=aa+bb-inter
    return inter/den if den>0 else 0.0

def center_wh(box):
    x1,y1,x2,y2=map(float,box)
    w=max(1e-9,x2-x1); h=max(1e-9,y2-y1)
    return (x1+x2)/2,(y1+y2)/2,w,h

def box_from_center(cx,cy,w,h):
    return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]

GT_BY_IMG = {k:g.reset_index(drop=True) for k,g in gt.groupby("sha256")}
PRED_BY_IMG = {k:g.reset_index(drop=True) for k,g in pred.groupby("sha256")}

matched_rows=[]
miss_rows=[]

for sha,gimg in GT_BY_IMG.items():
    pimg=PRED_BY_IMG.get(sha, pd.DataFrame(columns=pred.columns))

    for cid,gc in gimg.groupby("class_id"):
        gc=gc.reset_index(drop=True)
        pc=pimg[pimg.class_id==cid].reset_index(drop=True)

        if len(pc)==0:
            for _,gr in gc.iterrows():
                miss_rows.append({
                    "sha256":sha,
                    "class_id":int(cid),
                    "class":NAMES[int(cid)],
                    "reason":"NO_SAME_CLASS_PRED",
                    "image_path":gr.image_path,
                })
            continue

        G=len(gc); P=len(pc)
        cost=np.zeros((G,P),dtype=float)
        aux={}

        for i,gr in gc.iterrows():
            gb=[gr.gt_x1,gr.gt_y1,gr.gt_x2,gr.gt_y2]
            gcx,gcy,gw,gh=center_wh(gb)
            gdiag=max(1.0,math.hypot(gw,gh))

            for j,pr in pc.iterrows():
                pb=[pr.pred_x1,pr.pred_y1,pr.pred_x2,pr.pred_y2]
                pcx,pcy,pw,ph=center_wh(pb)
                cen=math.hypot(pcx-gcx,pcy-gcy)
                cen_diag=cen/gdiag
                ov=box_iou_xyxy(gb,pb)

                # center dominates; IoU + confidence only break ambiguous assignments.
                c=0.75*min(cen_diag,3.0)/3.0 + 0.20*(1.0-ov) + 0.05*(1.0-float(pr.conf))
                cost[i,j]=c
                aux[(i,j)]=(cen,cen_diag,ov)

        gi,pj=linear_sum_assignment(cost)

        paired_g=set()

        for i,j in zip(gi,pj):
            gr=gc.iloc[i]
            pr=pc.iloc[j]
            cen,cen_diag,actual_iou=aux[(i,j)]

            if not (cen_diag<=MAX_CENTER_DIAG_FOR_MATCH or actual_iou>=0.10):
                continue

            paired_g.add(i)

            gb=[gr.gt_x1,gr.gt_y1,gr.gt_x2,gr.gt_y2]
            pb=[pr.pred_x1,pr.pred_y1,pr.pred_x2,pr.pred_y2]

            gcx,gcy,gw,gh=center_wh(gb)
            pcx,pcy,pw,ph=center_wh(pb)

            pred_center_gt_extent=box_from_center(pcx,pcy,gw,gh)
            gt_center_pred_extent=box_from_center(gcx,gcy,pw,ph)

            center_only_iou=box_iou_xyxy(gb,pred_center_gt_extent)
            extent_only_iou=box_iou_xyxy(gb,gt_center_pred_extent)

            area_ratio=(pw*ph)/(gw*gh+1e-9)
            width_ratio=pw/(gw+1e-9)
            height_ratio=ph/(gh+1e-9)
            gt_ar=gw/(gh+1e-9)
            pred_ar=pw/(ph+1e-9)
            aspect_ratio_ratio=pred_ar/(gt_ar+1e-9)

            if actual_iou>=0.75:
                diagnosis="GOOD_075"
            elif actual_iou>=0.50:
                diagnosis="OK_050_075"
            elif center_only_iou>=0.75:
                diagnosis="CENTER_GOOD_EXTENT_BAD"
            elif center_only_iou>=0.50:
                diagnosis="CENTER_OK_EXTENT_OR_MIXED"
            else:
                diagnosis="CENTER_BAD_OR_WRONG_INSTANCE"

            matched_rows.append({
                "sha256":sha,
                "image_path":gr.image_path,
                "class_id":int(cid),
                "class":NAMES[int(cid)],
                "conf":float(pr.conf),

                "gt_x1":gr.gt_x1,"gt_y1":gr.gt_y1,
                "gt_x2":gr.gt_x2,"gt_y2":gr.gt_y2,

                "pred_x1":pr.pred_x1,"pred_y1":pr.pred_y1,
                "pred_x2":pr.pred_x2,"pred_y2":pr.pred_y2,

                "image_w":int(gr.image_w),
                "image_h":int(gr.image_h),
                "scale_1024":float(gr.scale_1024),

                "gt_w":gw,"gt_h":gh,
                "pred_w":pw,"pred_h":ph,

                "actual_iou":actual_iou,
                "center_error_px":cen,
                "center_error_gt_diag":cen_diag,
                "center_only_iou":center_only_iou,
                "extent_only_iou":extent_only_iou,

                "width_ratio":width_ratio,
                "height_ratio":height_ratio,
                "area_ratio":area_ratio,
                "aspect_ratio_ratio":aspect_ratio_ratio,
                "diagnosis":diagnosis,
            })

        for i,gr in gc.iterrows():
            if i not in paired_g:
                miss_rows.append({
                    "sha256":sha,
                    "class_id":int(cid),
                    "class":NAMES[int(cid)],
                    "reason":"NO_ACCEPTED_CENTER_AWARE_MATCH",
                    "image_path":gr.image_path,
                })

matched=pd.DataFrame(matched_rows)
misses=pd.DataFrame(miss_rows)

matched.to_csv(REPORT/"matched_localization_decomposition.csv.gz",index=False,compression="gzip")
misses.to_csv(REPORT/"unmatched_gt.csv.gz",index=False,compression="gzip")

print("Matched:",len(matched))
print("Unmatched GT:",len(misses))
display(matched["diagnosis"].value_counts().rename_axis("diagnosis").reset_index(name="count"))


## 9. Per-class center-vs-extent diagnosis

In [ ]:
def pass_rate(s,thr):
    return float((s>=thr).mean()) if len(s) else np.nan

summary_rows=[]

for cls,g in matched.groupby("class"):
    n=len(g)
    summary_rows.append({
        "class":cls,
        "matched":n,
        "actual_iou_median":float(g.actual_iou.median()),
        "center_only_iou_median":float(g.center_only_iou.median()),
        "extent_only_iou_median":float(g.extent_only_iou.median()),
        "center_error_diag_median":float(g.center_error_gt_diag.median()),

        "actual_pass50":pass_rate(g.actual_iou,.50),
        "actual_pass75":pass_rate(g.actual_iou,.75),
        "actual_pass90":pass_rate(g.actual_iou,.90),

        "center_only_pass50":pass_rate(g.center_only_iou,.50),
        "center_only_pass75":pass_rate(g.center_only_iou,.75),
        "center_only_pass90":pass_rate(g.center_only_iou,.90),

        "extent_only_pass50":pass_rate(g.extent_only_iou,.50),
        "extent_only_pass75":pass_rate(g.extent_only_iou,.75),
        "extent_only_pass90":pass_rate(g.extent_only_iou,.90),

        "center_good_extent_bad_rate":float((g.diagnosis=="CENTER_GOOD_EXTENT_BAD").mean()),
        "median_area_ratio_pred_gt":float(g.area_ratio.median()),
        "median_width_ratio":float(g.width_ratio.median()),
        "median_height_ratio":float(g.height_ratio.median()),
    })

decomp=pd.DataFrame(summary_rows).sort_values("class")
display(decomp)
decomp.to_csv(REPORT/"per_class_localization_decomposition.csv",index=False)

print("\nInterpretation:")
print("- center_only >> actual  => center is mostly right; bbox extent is hurting IoU")
print("- extent_only >> actual  => predicted size/shape is okay; center shift is hurting IoU")
print("- both low               => localization itself is genuinely poor / wrong instance")


## 10. Canonical ROI what-if sweep

For each matched GT in the three defect classes:

```text
canonical width  = max(GT width  × expand_factor, minimum_side_at_1024)
canonical height = max(GT height × expand_factor, minimum_side_at_1024)
```

The prediction is then given the **same canonical width/height but keeps its own predicted center**.

So this metric asks:

> If future training learned a standardized ROI extent perfectly, would the current predicted
> center already be good enough?

This deliberately isolates the center hypothesis. It is **not AP**.


In [ ]:
def clip_box(box,Wd,Hd):
    x1,y1,x2,y2=box
    return [
        max(0.0,min(float(Wd),x1)),
        max(0.0,min(float(Hd),y1)),
        max(0.0,min(float(Wd),x2)),
        max(0.0,min(float(Hd),y2)),
    ]

def canonical_dims(gt_w,gt_h,scale_1024,min_side_input_px,expand):
    # Convert the desired minimum side from 1024-input pixels to source-image pixels.
    min_side_orig=min_side_input_px/max(scale_1024,1e-9)
    cw=max(gt_w*expand,min_side_orig)
    ch=max(gt_h*expand,min_side_orig)
    return cw,ch

sweep_rows=[]

roi_matched=matched[matched["class"].isin(ROI_CLASSES)].copy()

for (cls,min_px,expand),dummy in [
    ((cls,min_px,expand),None)
    for cls in ROI_CLASSES
    for min_px in MIN_SIDE_INPUT_PX
    for expand in EXPAND_FACTORS
]:
    g=roi_matched[roi_matched["class"]==cls]
    if len(g)==0:
        continue

    vals=[]
    expansion=[]

    for _,r in g.iterrows():
        gcx=(r.gt_x1+r.gt_x2)/2
        gcy=(r.gt_y1+r.gt_y2)/2
        pcx=(r.pred_x1+r.pred_x2)/2
        pcy=(r.pred_y1+r.pred_y2)/2

        cw,ch=canonical_dims(
            r.gt_w,r.gt_h,r.scale_1024,
            min_px,expand
        )

        gt_can=clip_box(
            box_from_center(gcx,gcy,cw,ch),
            r.image_w,r.image_h
        )
        pred_can=clip_box(
            box_from_center(pcx,pcy,cw,ch),
            r.image_w,r.image_h
        )

        vals.append(box_iou_xyxy(gt_can,pred_can))

        gt_area=max(1e-9,r.gt_w*r.gt_h)
        can_area=max(0,gt_can[2]-gt_can[0])*max(0,gt_can[3]-gt_can[1])
        expansion.append(can_area/gt_area)

    vals=np.asarray(vals,float)
    expansion=np.asarray(expansion,float)

    actual=g.actual_iou.to_numpy(float)

    sweep_rows.append({
        "class":cls,
        "min_side_input_px":min_px,
        "expand_factor":expand,
        "n":len(g),

        "actual_median_iou":float(np.median(actual)),
        "canonical_median_iou":float(np.median(vals)),

        "actual_pass50":float((actual>=.50).mean()),
        "canonical_pass50":float((vals>=.50).mean()),
        "gain_pass50":float((vals>=.50).mean()-(actual>=.50).mean()),

        "actual_pass75":float((actual>=.75).mean()),
        "canonical_pass75":float((vals>=.75).mean()),
        "gain_pass75":float((vals>=.75).mean()-(actual>=.75).mean()),

        "actual_pass90":float((actual>=.90).mean()),
        "canonical_pass90":float((vals>=.90).mean()),
        "gain_pass90":float((vals>=.90).mean()-(actual>=.90).mean()),

        "median_roi_area_ratio_vs_original_gt":float(np.median(expansion)),
        "p90_roi_area_ratio_vs_original_gt":float(np.quantile(expansion,.90)),
    })

sweep=pd.DataFrame(sweep_rows)
sweep.to_csv(REPORT/"canonical_roi_sweep.csv",index=False)

# Show top balanced candidates: prioritize IoU@.75 gain but penalize huge boxes.
ranked=sweep.copy()
ranked["balanced_score"] = (
    ranked["gain_pass75"]
    - 0.035*np.maximum(0,ranked["median_roi_area_ratio_vs_original_gt"]-1.0)
)
ranked=ranked.sort_values(
    ["class","balanced_score","canonical_pass75"],
    ascending=[True,False,False]
)

print("Top canonical ROI candidates per class:")
display(ranked.groupby("class",group_keys=False).head(8))
ranked.to_csv(REPORT/"canonical_roi_ranked.csv",index=False)


## 11. Directly test the "20 px minimum" idea

This table isolates exactly the user's example: **minimum 20 px at 1024 input scale**.

It compares no expansion (`1.0×`) and moderate context (`1.2×`, `1.4×`).


In [ ]:
twenty=sweep[
    (sweep.min_side_input_px==20) &
    (sweep.expand_factor.isin([1.0,1.2,1.4]))
].copy()

display(twenty.sort_values(["class","expand_factor"]))
twenty.to_csv(REPORT/"canonical_roi_20px_focus.csv",index=False)


## 12. BBox scale-stratified diagnosis

This checks whether tiny/small boxes are actually the ones producing the center-vs-extent pattern.

Scale bins use normalized GT area:

- tiny `<0.1%`
- small `0.1–0.5%`
- medium `0.5–2%`
- large `>2%`


In [ ]:
def scale_bin(area_norm):
    if area_norm < .001:
        return "tiny_<0.1%"
    if area_norm < .005:
        return "small_0.1-0.5%"
    if area_norm < .02:
        return "medium_0.5-2%"
    return "large_>2%"

matched["gt_area_norm"] = (
    (matched.gt_w*matched.gt_h) /
    (matched.image_w*matched.image_h)
)
matched["scale_bin"] = matched.gt_area_norm.map(scale_bin)

scale_rows=[]

for (cls,bin_name),g in matched.groupby(["class","scale_bin"]):
    scale_rows.append({
        "class":cls,
        "scale_bin":bin_name,
        "n":len(g),
        "actual_iou_median":float(g.actual_iou.median()),
        "center_only_iou_median":float(g.center_only_iou.median()),
        "extent_only_iou_median":float(g.extent_only_iou.median()),
        "actual_pass75":pass_rate(g.actual_iou,.75),
        "center_only_pass75":pass_rate(g.center_only_iou,.75),
        "center_good_extent_bad_rate":float((g.diagnosis=="CENTER_GOOD_EXTENT_BAD").mean()),
    })

scale_diag=pd.DataFrame(scale_rows)
display(scale_diag.sort_values(["class","scale_bin"]))
scale_diag.to_csv(REPORT/"scale_stratified_localization.csv",index=False)


## 13. Visual review: center-good / extent-bad cases

Red = GT  
Blue = current prediction  
Green = hypothetical canonical ROI using the selected preview policy.

Default preview policy is deliberately modest:

- minimum side = 20 px at 1024 input scale;
- expansion = 1.20×.

This is only for visual judgment.


In [ ]:
PREVIEW_MIN_PX=20
PREVIEW_EXPAND=1.20
rng=np.random.default_rng(SEED)

def render_case(r):
    with Image.open(r.image_path) as im:
        im=ImageOps.exif_transpose(im).convert("RGB")
        d=ImageDraw.Draw(im)

        gt_box=[r.gt_x1,r.gt_y1,r.gt_x2,r.gt_y2]
        pred_box=[r.pred_x1,r.pred_y1,r.pred_x2,r.pred_y2]

        d.rectangle(gt_box,outline="red",width=max(2,int(min(im.size)/350)))
        d.rectangle(pred_box,outline="blue",width=max(2,int(min(im.size)/350)))

        gcx=(r.gt_x1+r.gt_x2)/2
        gcy=(r.gt_y1+r.gt_y2)/2
        cw,ch=canonical_dims(
            r.gt_w,r.gt_h,r.scale_1024,
            PREVIEW_MIN_PX,PREVIEW_EXPAND
        )
        can=clip_box(box_from_center(gcx,gcy,cw,ch),r.image_w,r.image_h)
        d.rectangle(can,outline="green",width=max(2,int(min(im.size)/350)))

        label=(
            f"{r['class']} actual={r.actual_iou:.2f} "
            f"center-only={r.center_only_iou:.2f}"
        )
        d.rectangle([0,0,min(im.width,850),34],fill="black")
        d.text((8,9),label,fill="white")

        im.thumbnail((420,320))
        canvas=Image.new("RGB",(420,320))
        canvas.paste(im,((420-im.width)//2,(320-im.height)//2))
        return canvas

sheet_paths=[]

for cls in ROI_CLASSES:
    pool=matched[
        (matched["class"]==cls) &
        (matched.diagnosis=="CENTER_GOOD_EXTENT_BAD")
    ].copy()

    if len(pool)==0:
        continue

    # Prioritize strong center evidence + poor actual IoU.
    pool["priority"]=pool.center_only_iou-pool.actual_iou
    pool=pool.sort_values("priority",ascending=False).head(40)

    ims=[render_case(r) for _,r in pool.iterrows()]
    cols=4
    rows=math.ceil(len(ims)/cols)
    sheet=Image.new("RGB",(cols*420,rows*320))

    for i,im in enumerate(ims):
        sheet.paste(im,((i%cols)*420,(i//cols)*320))

    p=PREVIEW/f"{cls}_center_good_extent_bad.jpg"
    sheet.save(p,quality=90)
    sheet_paths.append(p)

print("Preview sheets:", [str(p) for p in sheet_paths])


## 14. Decision report

The notebook does **not** automatically rewrite V6.2E.

Decision logic:

- if `center_only_iou` is much better than `actual_iou`, and canonical ROI pass@75 improves
  substantially with only modest area growth, then V6.3 should adopt a canonical ROI policy;
- if center-only is also poor, enlarging boxes would merely hide a real localization failure;
- if only tiny/small bins benefit, apply a minimum-context rule only below a scale threshold;
- if large `broken_strand` boxes remain extent-limited, inspect source annotation convention
  instead of making them even larger.


In [ ]:
decision_rows=[]

for cls in ROI_CLASSES:
    g=matched[matched["class"]==cls]
    if len(g)==0:
        continue

    cls_sweep=ranked[ranked["class"]==cls].copy()
    modest=cls_sweep[
        cls_sweep["median_roi_area_ratio_vs_original_gt"] <= 2.0
    ]
    best=(modest if len(modest) else cls_sweep).iloc[0]

    actual75=pass_rate(g.actual_iou,.75)
    center75=pass_rate(g.center_only_iou,.75)
    extent75=pass_rate(g.extent_only_iou,.75)

    center_gain=center75-actual75
    canonical_gain=float(best.gain_pass75)

    if center_gain >= .15 and canonical_gain >= .10:
        verdict="SUPPORTS_CANONICAL_ROI"
    elif center_gain >= .08:
        verdict="PARTIAL_SUPPORT_REVIEW_VISUALLY"
    else:
        verdict="DO_NOT_FIX_BY_ENLARGING_BOXES_ALONE"

    decision_rows.append({
        "class":cls,
        "matched":len(g),
        "actual_pass75":actual75,
        "center_only_pass75":center75,
        "extent_only_pass75":extent75,
        "center_only_gain75":center_gain,
        "center_good_extent_bad_rate":float((g.diagnosis=="CENTER_GOOD_EXTENT_BAD").mean()),
        "recommended_min_side_input_px":int(best.min_side_input_px),
        "recommended_expand_factor":float(best.expand_factor),
        "canonical_gain75":canonical_gain,
        "median_roi_area_ratio":float(best.median_roi_area_ratio_vs_original_gt),
        "verdict":verdict,
    })

decision=pd.DataFrame(decision_rows)
display(decision)
decision.to_csv(REPORT/"decision_summary.csv",index=False)

summary={
    "dataset_tag":FINAL_TAG,
    "model":"YOLO11m_v6p2e_1024",
    "model_sha256":MODEL_SHA,
    "split":"val",
    "test_touched":False,
    "matched_gt":int(len(matched)),
    "unmatched_gt":int(len(misses)),
    "roi_classes":ROI_CLASSES,
    "decision":decision.to_dict("records"),
}

(REPORT/"summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2)[:8000])


## 15. Package + upload report to Camber

In [ ]:
ZIP_OUT=W/f"{FINAL_TAG}_localization_canonical_roi_audit.zip"

with zipfile.ZipFile(ZIP_OUT,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(OUT.parent))

stash_put(ZIP_OUT,f"{REMOTE_REPORT}/{ZIP_OUT.name}")
stash_put(REPORT/"summary.json",f"{REMOTE_REPORT}/summary.json")
stash_put(REPORT/"decision_summary.csv",f"{REMOTE_REPORT}/decision_summary.csv")
stash_put(REPORT/"per_class_localization_decomposition.csv",
          f"{REMOTE_REPORT}/per_class_localization_decomposition.csv")
stash_put(REPORT/"canonical_roi_sweep.csv",
          f"{REMOTE_REPORT}/canonical_roi_sweep.csv")

print("✅ Audit complete.")
print("Report ZIP:",ZIP_OUT)
print("TEST was not predicted/evaluated.")
